# Production systems: a beginner’s tutorial
### Facts, rules, forward chaining, backward chaining, and a world that changes

This notebook uses the teacher’s **family relationships** and **pick-and-place** examples from the Knowledge Representation & Reasoning assignment. No prior knowledge of logic or reasoning systems is assumed. Some basic Python—tuples, lists, dictionaries, loops, and functions—is helpful.

**The engine is already implemented.** We import its functions rather than reimplementing them. Facts, rules, and actions are supplied as data in this notebook. In your submitted project, move those domain definitions into `domains.py` and import them here.

Work from top to bottom. Before running an example, try to predict its result; then compare it with the explanation underneath. Each main demonstration starts with fresh working memory, so earlier examples do not quietly supply its answers.

**Source and interface:** the attached assignment PDF and starter module. The attachment contains unfinished reasoners, so this tutorial assumes your completed engine follows the starter’s signatures. `applicable` is allowed to choose its own result format; our examples assume dictionaries with `name`, `add`, and either `terminate` or `delete`, as explained in Section 12. Project-specific helpers not present in the starter are not assumed.

## Route through the notebook

1. What a production system is
2. Setup and engine imports
3. Facts, patterns, variables, and bindings
4. Matching: `is_var`, `unify`, `substitute`, and the internal `_walk`
5. Managing knowledge with `WorkingMemory`
6. Writing the teacher’s family rules
7. Forward chaining: discovering conclusions
8. Backward chaining: answering questions
9. Comparing the two directions and understanding their limits
10. Time, actions, and the frame problem
11. Describing the hand-and-objects world
12. Finding applicable actions
13. Why recording only effects loses the rest of the world
14. Advancing time with persistence
15. A complete grab–move–release run
16. Function reference, exercises, and worked answers


## 1. What is a production system?

A production system combines **facts**, **rules**, and an **engine** that matches the rules to the facts.

Suppose we know:

- Mary is Ann’s parent.
- Ann is Sam’s parent.

And we have this rule:

> IF someone is the parent of a second person, AND that second person is the parent of a third person, THEN the first person is the grandparent of the third person.

The engine can conclude that **Mary is Sam’s grandparent**, even though we did not enter that fact ourselves.

| Part | Meaning | Family example |
|---|---|---|
| Working memory (WM) | Facts currently stored | Mary is Ann’s parent |
| Rule base | General relationships | A parent of a parent is a grandparent |
| Matching | Find values that make conditions hold together | First person = Mary, second = Ann, third = Sam |
| Firing a rule | Use those values to produce its conclusion | Mary is Sam’s grandparent |

A **production rule** is an IF–THEN rule. In Task 1, firing it adds knowledge. In Task 2, firing an action changes the represented world at the next time step.

The engine does not understand the everyday meaning of “parent” or “grab.” Those are labels in our data. It follows the structures and rules we supply. That is why the same engine can work with both family relationships and moving objects.


## 2. Setup: use your completed engine

This notebook assumes your existing package is called `reasoning` and its engine is `reasoning/engine.py`, or `src/reasoning/engine.py` in a project using a `src` layout.

1. Open this notebook in VS Code or Jupyter.
2. Select the Python environment that contains your project.
3. If the project has `pyproject.toml` or `setup.py`, install it from the project root with `python -m pip install -e .` using that environment. Restart the notebook kernel if necessary.
4. Run the import cell below.

For a standalone `engine.py` available on Python’s import path, change `from reasoning.engine` to `from engine`. Do not copy the unfinished starter over your completed engine.

The notebook’s examples use only Python’s standard library and your engine. Jupyter/VS Code is the notebook runner, not a reasoning or planning library.


In [ ]:
from reasoning.engine import (
    is_var, unify, substitute, WorkingMemory,
    forward_chain, backward_chain, applicable, step,
)

print("Engine functions imported successfully.")


If this raises `ModuleNotFoundError`, check the selected environment and package installation. If it raises `ImportError`, check that the named functions are exported by your engine. If a later cell raises `NotImplementedError`, that function still contains the starter stub.

**Contract used here:** `forward_chain(wm, rules)` updates and returns WM; `backward_chain(wm, rules, query, subst=None)` yields bindings; `applicable(wm, action, t)` yields grounded actions; `step(wm, grounded_action, t)` updates and returns WM while retaining history. If your implementation uses another interface, adjust the relevant calls to your documented interface.


## 3. Facts, patterns, variables, and bindings

A fact is represented as a **flat tuple**. “Flat” means that its items are individual values, rather than nested expressions.

```python
("parent", "mary", "ann")
```

Read this as “Mary is a parent of Ann.” The first item, `"parent"`, is the **predicate**: the relationship being described. The remaining items are its **arguments**. Their order matters: `parent(mary, ann)` and `parent(ann, mary)` say different things.

A **constant** names a particular value, such as `"mary"`, `"A"`, or `0`. A **variable** is a string starting with `?`, such as `"?child"`.

```python
("parent", "mary", "?child")
```

This is a **pattern** asking who Mary’s children are. A **binding** supplies a value for a variable:

```python
{"?child": "ann"}
```

The engine calls a dictionary of bindings a **substitution**, often abbreviated `subst`. A **ground** fact contains no variables. Store ground facts in working memory; put variables in queries, rules, and action templates.

These labels are case-sensitive. `"mary"` and `"Mary"` are different constants. Predicate spelling and tuple length must also agree.


In [ ]:
fact = ("parent", "mary", "ann")
pattern = ("parent", "mary", "?child")
bindings = {"?child": "ann"}

print("Fact:", fact)
print("Pattern:", pattern)
print("Bindings:", bindings)


## 4. The matching machinery

### 4.1 `is_var(term)`: is this a variable?

The convention is entirely syntactic: a string starting with `?` is a variable. A name like `"x"` without `?` is a constant.


In [ ]:
for term in ["?child", "child", "mary", 0]:
    print(repr(term), "->", is_var(term))


Expected: only `"?child"` produces `True`.

### 4.2 `unify(pattern, fact, subst)`: can these tuples match?

**Unification** finds consistent bindings that make two tuples identical. Pass `{}` or `None` when you have no existing bindings. Either tuple can contain variables.

It returns a binding dictionary on success, or `None` on failure. The supplied dictionary is not changed.


In [ ]:
b = unify(("parent", "?parent", "?child"), ("parent", "mary", "ann"), {})
print(b)
assert b == {"?parent": "mary", "?child": "ann"}

print("Different constants:", unify(("parent", "mary", "sam"), fact, {}))
print("Different lengths:", unify(("parent", "?x"), fact, {}))


The first match succeeds with `?parent = mary` and `?child = ann`. The other two return `None`.

**An empty dictionary can still mean success.** A match between identical ground facts requires no variable bindings, so it returns `{}`. Test `result is not None`, rather than `if result`, when testing unification success.


In [ ]:
result = unify(("parent", "mary", "ann"), ("parent", "mary", "ann"), {})
print("Result:", result)
print("Successful match?", result is not None)
assert result == {}


One variable must have one consistent value throughout a match. Existing bindings must also be respected.


In [ ]:
print(unify(("parent", "?x", "?x"), ("parent", "mary", "ann"), {}))
# None: the same ?x cannot be both mary and ann.

existing = {"?x": "mary"}
print(unify(("parent", "?x", "?y"), ("parent", "mary", "ann"), existing))
print("Original bindings:", existing)
assert existing == {"?x": "mary"}

print(unify(("parent", "?x", "?y"), ("parent", "ann", "sam"), existing))
# None: ?x was already fixed to mary.


### 4.3 `substitute(pattern, subst)`: put the values into a tuple

Unification **finds** bindings. Substitution **uses** them. This is how the engine turns a rule’s general conclusion into a particular fact.


In [ ]:
b = {"?x": "mary", "?z": "sam"}
print(substitute(("grandparent", "?x", "?z"), b))
print(substitute(("parent", "?x", "?unbound"), b))


Expected: `('grandparent', 'mary', 'sam')`, then `('parent', 'mary', '?unbound')`. Unbound variables remain variables; substitution alone does not discover their values.

### 4.4 Internal helper `_walk(term, subst)`

Sometimes a binding points to another variable: `?x → ?y → mary`. `_walk` follows that chain to its final value. The leading underscore marks it as an internal helper; normal client code uses `substitute`, which already calls it. This example demonstrates its effect through the public interface, so you do not need to import a private function.


In [ ]:
chain = {"?x": "?y", "?y": "mary"}
print(substitute(("parent", "?x", "ann"), chain))
# Internally, _walk("?x", chain) returns "mary".


## 5. Working memory: storing and looking up facts

### 5.1 `WorkingMemory(facts=())` and `.facts`

Working memory stores a **set** of facts: duplicates disappear, and iteration order is unspecified. Use `sorted(...)` to make these examples easier to read. Treat `.facts` as read-only in client code; use methods to update it.


In [ ]:
wm = WorkingMemory([
    ("parent", "mary", "ann"),
    ("parent", "ann", "sam"),
])
print(wm)
print(sorted(wm.facts))


`print(wm)` uses the class’s `__repr__` method and displays `WorkingMemory(2 facts)`. It is a compact description, not a list of the facts.

### 5.2 `.tell(fact)` and `.retract(fact)`

`tell` adds a fact. `retract` removes that exact fact. Adding an existing fact or removing a missing one is harmless.


In [ ]:
wm.tell(("parent", "mary", "bob"))
wm.tell(("parent", "mary", "bob"))
print("After adding twice:", sorted(wm.facts))
assert len(wm.facts) == 3

wm.retract(("parent", "mary", "bob"))
wm.retract(("parent", "mary", "bob"))
print("After removing twice:", sorted(wm.facts))
assert len(wm.facts) == 2


### 5.3 `.holds(fact)`: test exact membership

`holds` checks whether an exact ground fact is stored. It does not reason with rules and does not interpret variables as wildcards.


In [ ]:
print(wm.holds(("parent", "mary", "ann")))       # True
print(wm.holds(("grandparent", "mary", "sam"))) # False: not stored yet
print(wm.holds(("parent", "mary", "?child")))   # False: use ask for patterns


### 5.4 `.ask(pattern, subst=None)`: find matching stored facts

`ask` yields one binding dictionary for each matching fact. A function that **yields** results gives us an iterator; `list(...)` collects all its results, and a `for` loop consumes them one at a time.

`ask` only searches facts already in working memory. It does not use the rule base.


In [ ]:
print(list(wm.ask(("parent", "mary", "?child"))))
print(sorted(b["?child"] for b in wm.ask(("parent", "mary", "?child"))))

print(list(wm.ask(("parent", "mary", "ann")))) # [{}]: one successful match
print(list(wm.ask(("parent", "mary", "sam")))) # []: no match

print(list(wm.ask(("parent", "?x", "?y"), {"?x": "ann"})))
# [{'?x': 'ann', '?y': 'sam'}]


**Be careful:** `[{}]` means one match with no new bindings. `[]` means no match. Also, `bool(wm.ask(...))` tests the iterator object, not whether a result exists. For an existence check, use `any(True for _ in wm.ask(pattern))`, or use `holds` for a ground fact.

### 5.5 `.snapshot()` and `.restore(snapshot)`: save and restore

A snapshot is a copy of the stored fact set. It lets you explore changes and then return to the saved memory. It does not save Python variables such as your own time counter or your rule list.


In [ ]:
saved = wm.snapshot()
wm.tell(("parent", "sam", "leo"))
print("During experiment:", sorted(wm.facts))
wm.restore(saved)
print("Restored:", sorted(wm.facts))
assert not wm.holds(("parent", "sam", "leo"))


## 6. The teacher’s family rules

We will use the original two parent facts and all three ancestry rules from the assignment.

A rule is a dictionary with an `"if"` list and a `"then"` tuple. Every condition in the list must hold **under compatible bindings**. This is an AND, not an OR. To express alternatives, supply separate rules.

The same variable name connects conditions **within a rule**. In the grandparent rule, `?y` must be the same person in both parent facts.


In [ ]:
FAMILY_FACTS = [
    ("parent", "mary", "ann"),
    ("parent", "ann", "sam"),
]

FAMILY_RULES = [
    {
        "if": [("parent", "?x", "?y"), ("parent", "?y", "?z")],
        "then": ("grandparent", "?x", "?z"),
    },
    {
        "if": [("parent", "?x", "?y")],
        "then": ("ancestor", "?x", "?y"),
    },
    {
        "if": [("parent", "?x", "?y"), ("ancestor", "?y", "?z")],
        "then": ("ancestor", "?x", "?z"),
    },
]


Read these rules aloud:

1. A parent of a parent is a grandparent.
2. Every parent is an ancestor.
3. A parent of someone who is an ancestor of a third person is also an ancestor of that third person.

The third rule is **recursive**: it uses `ancestor` to establish another `ancestor` relationship. This lets a small rule describe chains of different lengths.

### Follow one complete rule match

The next cell manually follows the grandparent rule’s conditions using `ask`. This illustrates the bindings; it is not another implementation of the reasoner.


In [ ]:
family = WorkingMemory(FAMILY_FACTS)
for first_binding in family.ask(("parent", "?x", "?y")):
    for complete_binding in family.ask(("parent", "?y", "?z"), first_binding):
        print("Matched:", complete_binding)
        print("Conclusion:", substitute(("grandparent", "?x", "?z"), complete_binding))


The complete match is `?x = mary`, `?y = ann`, `?z = sam`.

Trying the first condition with `?x = ann`, `?y = sam` does not yield a complete match: there is no stored fact saying Sam is a parent. One successful condition alone is not enough to fire the rule.


## 7. Forward chaining: “What follows from what I know?”

Forward chaining is **data-driven**. It starts with facts, matches rule conditions, adds conclusions, and repeats. A newly derived fact may help another rule match.

For this example:

- `parent(mary, ann)` and `parent(ann, sam)` establish `grandparent(mary, sam)`.
- The parent facts establish `ancestor(mary, ann)` and `ancestor(ann, sam)`.
- `parent(mary, ann)` together with `ancestor(ann, sam)` establishes `ancestor(mary, sam)`.

The precise order of discoveries depends on the engine’s iteration order. The final set should be the same for these rules.

### `forward_chain(wm, rules)`

It updates WM and returns it. A **fixpoint** is reached when a complete pass adds no new facts. No new facts means no new matches can produce additional conclusions.


In [ ]:
forward_wm = WorkingMemory(FAMILY_FACTS)
before = forward_wm.snapshot()
result = forward_chain(forward_wm, FAMILY_RULES)

print("Initial facts:")
for f in sorted(before):
    print(" ", f)
print("Derived facts:")
for f in sorted(forward_wm.facts - before):
    print(" ", f)
print("Total facts:", len(forward_wm.facts))

expected_derived = {
    ("grandparent", "mary", "sam"),
    ("ancestor", "mary", "ann"),
    ("ancestor", "ann", "sam"),
    ("ancestor", "mary", "sam"),
}
assert forward_wm.facts == set(FAMILY_FACTS) | expected_derived
assert result is forward_wm


Expected: **2 initial facts + 4 derived facts = 6 facts**. All original facts remain.

After forward chaining, ordinary lookup can find the stored conclusions. Running forward chaining again adds nothing.


In [ ]:
print(sorted(b["?who"] for b in forward_wm.ask(("ancestor", "mary", "?who"))))
# ['ann', 'sam']

at_fixpoint = forward_wm.snapshot()
forward_chain(forward_wm, FAMILY_RULES)
assert forward_wm.facts == at_fixpoint
print("A second run adds nothing: fixpoint reached.")


### Why Task 1 is monotonic

**Monotonic** means adding knowledge does not invalidate previously derived conclusions under these positive rules. Here, forward chaining only adds facts; no rule retracts one. The world is treated as a timeless collection of relationships.

The frame problem concerns what remains true across a change of state. Task 1 has no state transitions to represent, so it does not face that problem.

The `WorkingMemory` class still offers `retract`, but Task 1’s forward loop does not use it. If you manually remove a supporting fact **after** materialising conclusions, this simple engine does not automatically remove conclusions that depended on it. Rebuild WM from the remaining base facts and rerun forward chaining when you need that behaviour.


## 8. Backward chaining: “Can I establish this particular goal?”

Backward chaining is **goal-driven**. It begins with a question and works toward supporting facts.

For `grandparent(mary, sam)`:

1. Is that fact already stored? If yes, it is established directly.
2. Otherwise, find a rule whose conclusion matches the goal.
3. The grandparent rule requires `parent(mary, ?y)` AND `parent(?y, sam)`.
4. The first fact gives `?y = ann`; the second fact confirms `parent(ann, sam)`.
5. Both conditions hold consistently, so the goal is established.

This does **not** reverse the implication to conclude parents from a grandparent fact. It proves a proposed conclusion by establishing the rule’s premises.

### `backward_chain(wm, rules, query, subst=None)`

It yields substitutions for successful proofs. Use fresh WM containing only the original parent facts so we can see that the rules actually do the work.


In [ ]:
backward_wm = WorkingMemory(FAMILY_FACTS)
query = ("grandparent", "mary", "sam")
print("Already stored?", backward_wm.holds(query))
print("Can be proved?", any(True for _ in backward_chain(backward_wm, FAMILY_RULES, query)))
print("Unsupported goal?", any(True for _ in backward_chain(
    backward_wm, FAMILY_RULES, ("grandparent", "sam", "mary")
)))


Expected: `False`, `True`, `False`. “Unsupported” means the goal cannot be established from these facts and rules. It is not a proof of an explicit negative fact.

### Questions with variables

Use substitution to turn proof bindings into answers. A backward chainer may return extra bindings for fresh internal rule variables, and different proofs can yield the same answer. Project onto the original query and collect a set to display distinct answers.


In [ ]:
query = ("ancestor", "mary", "?who")
answers = {
    substitute(query, b)
    for b in backward_chain(backward_wm, FAMILY_RULES, query)
}
print(sorted(answers))
assert answers == {
    ("ancestor", "mary", "ann"),
    ("ancestor", "mary", "sam"),
}
print("WM after asking:", sorted(backward_wm.facts))
assert backward_wm.facts == set(FAMILY_FACTS)


Expected answers: Ann and Sam. Under this interface, backward chaining establishes answers without adding derived facts to WM.

### Following the recursive proof

To prove `ancestor(mary, sam)`, the recursive rule asks for:

- `parent(mary, ?middle)`, giving `?middle = ann`;
- `ancestor(ann, sam)`, which the direct-parent rule proves from `parent(ann, sam)`.

A **base case** reaches a stored fact. A **recursive case** creates another goal using the same relationship.

At each rule use, the engine must give its variables fresh names (**standardise apart**). The `?x` in one use of a rule must not accidentally become the same variable as the `?x` in another use. Users can keep writing readable names in the rule data; the engine handles the renaming.

### Supply existing bindings

The optional `subst` parameter restricts a query or continues an existing proof. Here we fix the starting person to Mary.


In [ ]:
query = ("ancestor", "?person", "?who")
restricted = {
    substitute(query, b)
    for b in backward_chain(backward_wm, FAMILY_RULES, query, {"?person": "mary"})
}
print(sorted(restricted))
assert restricted == answers


### Test a longer ancestry chain

Add one extra parent fact to exercise recursion beyond the original two generations. The rules remain unchanged.


In [ ]:
extended_wm = WorkingMemory(FAMILY_FACTS + [("parent", "sam", "leo")])
query = ("ancestor", "mary", "?who")
extended_answers = {
    substitute(query, b)
    for b in backward_chain(extended_wm, FAMILY_RULES, query)
}
print(sorted(extended_answers))
assert extended_answers == {
    ("ancestor", "mary", "ann"),
    ("ancestor", "mary", "sam"),
    ("ancestor", "mary", "leo"),
}


A recursion limit or another cycle-control strategy is needed to keep backward search from looping. A depth limit can also prevent discovery of valid proofs deeper than that limit. These small acyclic examples assume the engine’s bound is sufficient. Recursive rules do not guarantee termination by themselves.


## 9. Forward and backward chaining side by side

| Question | Forward chaining | Backward chaining |
|---|---|---|
| Where does it start? | Available facts | A query or goal |
| What does it try to do? | Derive all reachable conclusions | Establish the requested goal and bindings |
| Does it store derived facts here? | Yes | No |
| When is it useful? | Many later queries over the same knowledge | A particular question without deriving everything |
| Does it use the same rules? | Yes | Yes |

Both methods should agree on the distinct answers for these finite examples when backward search explores the necessary proofs.


In [ ]:
fc_wm = WorkingMemory(FAMILY_FACTS)
bc_wm = WorkingMemory(FAMILY_FACTS)
forward_chain(fc_wm, FAMILY_RULES)

for query in [
    ("ancestor", "mary", "?who"),
    ("grandparent", "?person", "sam"),
    ("ancestor", "sam", "mary"),
]:
    from_forward = {substitute(query, b) for b in fc_wm.ask(query)}
    from_backward = {substitute(query, b) for b in backward_chain(bc_wm, FAMILY_RULES, query)}
    print(query, "->", sorted(from_forward))
    assert from_forward == from_backward


The engine follows the rules you provide, not every relationship a human might assume. We did not define “sibling” or “mother,” so it cannot establish those relationships. Failure to prove a claim means **not established by this knowledge base**; it does not automatically mean the opposite is known.


## 10. Reasoning through time: when the world changes

Now consider a hand and two objects. Initially, the hand and object 1 are at A; object 2 is at C. We want object 1 to end up at B.

A hand can grab an object, move to another location, and release the object. Unlike a family inference, these actions **cause a different state of the world**.

We append a time step to **every fact**:

```python
("at", "obj1", "A", 0)    # object 1 is at A at step 0
("holding", "obj1", 1)    # the hand holds object 1 at step 1
```

The final integer is a discrete state index, not a duration in seconds. Conditions are checked at `t`; effects hold at `t + 1`. We perform one action per transition.

### The frame problem

After grabbing object 1, where is object 2? It should still be at C. The grab action does not mention object 2, but we still need that fact in the next state.

If we record **only effects**, object 2’s location—and even the hand’s location—will be absent at the next time step. Old facts remain in history, but a fact at time 0 is not automatically a fact at time 1.

We therefore use **persistence**: carry every fact at `t` forward to `t + 1`, unless the action terminates it. This generic rule also carries facts describing objects, locations, and connections.

Writing separate “this stays unchanged” statements for every action and unrelated fact would grow rapidly as the domain grows. Persistence states that default once in the engine, and actions describe their exceptions.


## 11. Describe the world and its actions as data

We retain the teacher’s grab example and complete the small world with move and release actions.

Our representation makes these choices:

- A held object is represented by `holding(obj)`. It has no separate `at(obj, location)` fact while being held. To find its physical location, use the hand’s location.
- Moving the hand therefore does not need to update a held object’s `at` fact.
- The `object` condition prevents the hand itself from matching `?o` in a grab.
- The `link` facts list permitted moves. The move schema does not assume every location is reachable from every other location.
- In the action schemas, conditions include `?t`; effect templates omit time. The engine adds the next time step.

All of these are domain choices expressed as data; none requires a predicate-specific branch in the engine.


In [ ]:
WORLD_FACTS = [
    ("at", "hand", "A", 0),
    ("at", "obj1", "A", 0),
    ("at", "obj2", "C", 0),
    ("empty", "hand", 0),
    ("object", "obj1", 0),
    ("object", "obj2", 0),
    ("location", "A", 0),
    ("location", "B", 0),
    ("location", "C", 0),
    ("link", "A", "B", 0),
    ("link", "B", "A", 0),
    ("link", "B", "C", 0),
    ("link", "C", "B", 0),
]

GRAB = {
    "name": ("grab", "?o", "?loc"),
    "if": [
        ("at", "hand", "?loc", "?t"),
        ("at", "?o", "?loc", "?t"),
        ("empty", "hand", "?t"),
        ("object", "?o", "?t"),
    ],
    "add": [("holding", "?o")],
    "delete": [("empty", "hand"), ("at", "?o", "?loc")],
}

MOVE = {
    "name": ("move", "?from", "?to"),
    "if": [
        ("at", "hand", "?from", "?t"),
        ("link", "?from", "?to", "?t"),
    ],
    "add": [("at", "hand", "?to")],
    "delete": [("at", "hand", "?from")],
}

RELEASE = {
    "name": ("release", "?o", "?loc"),
    "if": [
        ("holding", "?o", "?t"),
        ("at", "hand", "?loc", "?t"),
    ],
    "add": [("at", "?o", "?loc"), ("empty", "hand")],
    "delete": [("holding", "?o")],
}

ACTIONS = [GRAB, MOVE, RELEASE]


A **schema** is a reusable action template with variables. A **ground action instance** has specific values, such as `grab(obj1, A)`.

The starter uses `delete` in its example schema but suggests `terminate` in the grounded result. Here, schemas use `delete`; a grounded result may use either key as long as your `step` understands the result returned by your `applicable`. If your engine consistently uses another format, adjust these action definitions/calls together.


## 12. `applicable(wm, action_schema, t)`: what can happen now?

This function binds `?t` to the requested time and matches every precondition with consistent bindings. It yields ground action instances. It does **not** execute them.

For the following cells, a grounded result is a dictionary with:

| Key | Example | Meaning |
|---|---|---|
| `name` | `('grab', 'obj1', 'A')` | Which particular action |
| `add` | `[('holding', 'obj1')]` | Untimed facts to establish next |
| `terminate` or `delete` | `[('empty', 'hand'), ('at', 'obj1', 'A')]` | Untimed facts not to carry forward |

The teacher explicitly leaves the instance format to your implementation. If it yields a tuple or a custom object, this is the point to adapt the notebook to that interface.


In [ ]:
world = WorkingMemory(WORLD_FACTS)
initial_snapshot = world.snapshot()

grab_options = list(applicable(world, GRAB, 0))
print("Possible grabs:")
for action in grab_options:
    print(action)
assert {tuple(a["name"]) for a in grab_options} == {("grab", "obj1", "A")}
assert world.facts == initial_snapshot

print("Possible moves:", sorted(tuple(a["name"]) for a in applicable(world, MOVE, 0)))
print("Possible releases:", list(applicable(world, RELEASE, 0)))


Expected: grab object 1 at A; move from A to B; no release. Object 2 is at C, so the hand cannot grab it at step 0. The hand is empty, so it cannot release anything.

When there are several results, choose an action by its grounded name rather than relying on result order. The next function is a small **notebook convenience** for selection, not part of the engine and not a planner.


In [ ]:
def choose_action(wm, action_schema, t, desired_name):
    matches = [
        action for action in applicable(wm, action_schema, t)
        if tuple(action["name"]) == tuple(desired_name)
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one applicable {desired_name} at step {t}; got {len(matches)}")
    return matches[0]

grab_obj1 = choose_action(world, GRAB, 0, ("grab", "obj1", "A"))
print("Selected:", grab_obj1["name"])


## 13. First experiment: record only what the action adds

The assignment asks us to observe the failure before introducing persistence. We can illustrate it without implementing a second transition engine: create a separate WM containing the old history and only the grab action’s added facts at step 1.

This deliberately incomplete experiment omits the carry-forward rule. Keep it separate from the real world.


In [ ]:
effects_only = WorkingMemory(WORLD_FACTS)
for added_fact in grab_obj1["add"]:
    effects_only.tell(tuple(added_fact) + (1,))

print("Facts at step 1:", sorted(f for f in effects_only.facts if f[-1] == 1))
print("Holding obj1 at 1?", effects_only.holds(("holding", "obj1", 1)))
print("obj2 at C at 1?", effects_only.holds(("at", "obj2", "C", 1)))
print("Hand at A at 1?", effects_only.holds(("at", "hand", "A", 1)))
print("Possible moves at 1:", list(applicable(effects_only, MOVE, 1)))
assert effects_only.holds(("at", "obj2", "C", 0))
assert not effects_only.holds(("at", "obj2", "C", 1))


Only `holding(obj1, 1)` is recorded for step 1. We cannot establish where object 2 or the hand is **at that step**, and there is no applicable move. The engine has not inferred that object 2 was destroyed; its location simply was not represented in the next state.

Keeping the step-0 fact is insufficient because the action’s conditions ask for facts stamped with step 1.


## 14. `step(wm, grounded_action, t)`: advance with persistence

Let `Sₜ` be the set of **untimed** fact bodies at step `t`, `D` the terminated bodies, and `A` the added bodies. The next state is:

**Sₜ₊₁ = (Sₜ − D) ∪ A**

The engine stamps those next-state bodies with `t + 1` and keeps earlier time-stamped facts as history. It processes all predicates generically, including `object`, `location`, and `link`.

Select an applicable action first, then pass the grounded instance to `step`. The starter does not specify how `step` handles an invalid action, so this notebook does not assume it validates arbitrary input itself.


In [ ]:
world = WorkingMemory(WORLD_FACTS)
grab_obj1 = choose_action(world, GRAB, 0, ("grab", "obj1", "A"))
returned_wm = step(world, grab_obj1, 0)
assert returned_wm is world

print("All facts at step 1:")
for f in sorted(f for f in world.facts if f[-1] == 1):
    print(" ", f)

assert world.holds(("holding", "obj1", 1))
assert not world.holds(("empty", "hand", 1))
assert not world.holds(("at", "obj1", "A", 1))
assert world.holds(("at", "hand", "A", 1))
assert world.holds(("at", "obj2", "C", 1))
assert world.holds(("location", "B", 1))
assert world.holds(("object", "obj2", 1))
assert world.holds(("link", "A", "B", 1))
assert world.holds(("at", "obj1", "A", 0))  # historical fact retained


The action changes two current facts and adds one. All unaffected facts persist.

**Delete does not mean erase history.** Here it means “do not carry this fact into the next state.” `at(obj1, A, 0)` remains a valid historical fact even though `at(obj1, A, 1)` is absent. Calling `retract` on a historical tuple would remove that history, so it is not how we model this transition.

We can check the entire next state against the persistence equation, including predicates that are not about positions.


In [ ]:
initial_bodies = {f[:-1] for f in WORLD_FACTS if f[-1] == 0}
deleted_bodies = {
    tuple(f) for f in grab_obj1.get("terminate", grab_obj1.get("delete", []))
}
added_bodies = {tuple(f) for f in grab_obj1["add"]}
expected_bodies = (initial_bodies - deleted_bodies) | added_bodies
actual_bodies = {f[:-1] for f in world.facts if f[-1] == 1}
assert actual_bodies == expected_bodies
print("The whole next state obeys the persistence equation.")


## 15. A complete run: take object 1 from A to B

We choose a three-action sequence ourselves. Finding this sequence automatically would be **planning**, the optional harder task in the assignment; `step` only executes the selected transition.

| Step | Action just performed | Hand | Object 1 | Object 2 |
|---|---|---|---|---|
| 0 | Initial state | At A, empty | At A | At C |
| 1 | Grab obj1 at A | At A, holding obj1 | Held | At C |
| 2 | Move A → B | At B, holding obj1 | Held | At C |
| 3 | Release obj1 at B | At B, empty | At B | At C |

The display helper below is domain-specific notebook code. The engine remains generic. We display every fact at a selected time, so the persistence of locations and connections is visible too.


In [ ]:
def show_state(wm, t):
    print(f"State at step {t}:")
    for f in sorted(f for f in wm.facts if f[-1] == t):
        print(" ", f)
    print()

world = WorkingMemory(WORLD_FACTS)
sequence = [
    (GRAB, ("grab", "obj1", "A")),
    (MOVE, ("move", "A", "B")),
    (RELEASE, ("release", "obj1", "B")),
]

show_state(world, 0)
for t, (schema, action_name) in enumerate(sequence):
    selected = choose_action(world, schema, t, action_name)
    print("Perform:", selected["name"])
    step(world, selected, t)
    show_state(world, t + 1)

    # The untouched object and world descriptions survive every transition.
    assert world.holds(("at", "obj2", "C", t + 1))
    assert world.holds(("object", "obj2", t + 1))
    assert world.holds(("location", "C", t + 1))

assert world.holds(("at", "obj1", "B", 3))
assert world.holds(("at", "hand", "B", 3))
assert world.holds(("empty", "hand", 3))
assert not world.holds(("holding", "obj1", 3))
assert not world.holds(("at", "obj1", "A", 3))
print("Goal reached: obj1 is at B; obj2 stayed at C throughout.")


### Ask questions about history

Always include the time slot. Without it, a Task-1-style tuple has the wrong length for these facts. To ask across time, use a variable in that slot.


In [ ]:
print("Objects at C at step 2:",
      sorted(b["?o"] for b in world.ask(("at", "?o", "C", 2))))

print("obj1's recorded location facts:",
      sorted((b["?t"], b["?loc"]) for b in world.ask(("at", "obj1", "?loc", "?t"))))

print("Times when obj1 was held:",
      sorted(b["?t"] for b in world.ask(("holding", "obj1", "?t"))))


Expected: object 2 at C at step 2; object 1’s location records are `(0, A)` and `(3, B)`; it was held at steps 1 and 2. During holding, its location is represented indirectly by the hand’s location.

### Explore a branch, then restore it

Take a snapshot before exploring a new transition. If you try an alternative transition from an old time without first restoring its snapshot, you can accidentally put different possible worlds into the same time-stamped memory. Time stamps alone do not identify branches.


In [ ]:
saved_world = world.snapshot()
saved_time = 3

move_back = choose_action(world, MOVE, saved_time, ("move", "B", "A"))
step(world, move_back, saved_time)
print("Explored hand at A at step 4:", world.holds(("at", "hand", "A", 4)))

world.restore(saved_world)
current_time = saved_time
assert world.facts == saved_world
assert not any(f[-1] == 4 for f in world.facts)
print("Restored world and time to step", current_time)


### Inference versus causation

Deriving `grandparent(mary, sam)` does not make Mary become Sam’s grandparent. It makes an existing logical consequence explicit. In contrast, executing `grab(obj1, A)` changes the represented state: the hand is empty before the action and holds object 1 afterward.

The time ordering `t → t + 1` makes cause precede effect. The domain model says what the action causes; time ordering alone does not establish causality from observations.

Although this engine retains historical facts and so its total stored set grows, the **current world state changes**: facts can cease to hold at later steps. Separate “the archive grows” from “the current state only gains truths.”


## 16. Function reference

### Matching and memory

| Function or method | Use | Result or effect |
|---|---|---|
| `is_var(term)` | Recognise a `?` variable | Boolean |
| `unify(a, b, subst)` | Find consistent bindings between flat tuples | New dictionary or `None` |
| `substitute(pattern, subst)` | Resolve bound variables | A tuple; unbound variables remain |
| `_walk(term, subst)` | Internal variable-chain resolution | Final bound value, or unbound variable |
| `WorkingMemory(facts=())` | Create a memory | A WM instance |
| `wm.tell(fact)` | Assert one ground fact | Adds to WM |
| `wm.retract(fact)` | Remove one exact stored fact | Removes from WM if present |
| `wm.holds(fact)` | Check exact stored membership | Boolean |
| `wm.ask(pattern, subst=None)` | Match stored facts | Iterator of binding dictionaries |
| `wm.snapshot()` | Save the current facts | Copy of the fact set |
| `wm.restore(snapshot)` | Reset the fact set | Replaces WM’s contents |
| `wm.facts` | Inspect stored facts | Set; treat as read-only |
| `wm.__repr__()` | Compact description; called by `print(wm)` | String such as `WorkingMemory(2 facts)` |

### Reasoning and time

| Engine function | Use | Result or effect |
|---|---|---|
| `forward_chain(wm, rules)` | Derive consequences to a fixpoint | Updates and returns WM |
| `backward_chain(wm, rules, query, subst=None)` | Establish a goal using facts and rules | Iterator of proof bindings |
| `applicable(wm, action_schema, t)` | Match action preconditions at `t` | Iterator of grounded instances |
| `step(wm, grounded_action, t)` | Apply effects and persistence | Updates and returns WM, retaining history |

`choose_action` and `show_state` are notebook helpers, not engine functions. Your engine may contain additional implementation helpers, such as a conjunction matcher, rule-variable freshener, or answer-display helper. Their signatures are not specified in the supplied starter. Their roles are illustrated here by condition matching, fresh rule variables, and projecting proof bindings onto queries.

### Common points of confusion

- **`ask` versus backward chaining:** `ask` searches stored facts. Backward chaining also tries rules.
- **`unify` versus `substitute`:** unify finds bindings; substitute fills them into a tuple.
- **`None`, `{}`, `[]`, and `[{}]`:** unification failure; successful match without bindings; no query results; one query result without bindings.
- **Schema versus instance:** a template has variables; an applicable instance has concrete values.
- **`applicable` versus `step`:** one discovers permitted actions; the other performs a selected transition.
- **Deleting versus history:** terminate a fact for the next state while preserving the past.
- **Proof versus plan:** a proof establishes a logical consequence; a plan is a sequence of actions that achieves a future state.
- **Persistence versus constraint checking:** persistence carries unchanged facts. It does not by itself prevent two objects being held or contradictory locations. Correct initial data, preconditions, and effects must maintain domain constraints.


## 17. Try it yourself

For each exercise, write down the expected result before running your code. Use fresh WM when you want an independent experiment.

1. What does `unify(("parent", "?p", "sam"), ("parent", "ann", "sam"), {})` return? What if `?p` is already bound to `mary`?
2. Why does `ask(("grandparent", "mary", "?who"))` initially return nothing, while backward chaining can answer the question?
3. Add `parent(sam, leo)` to the teacher’s family facts. List Mary’s descendants using each reasoning direction. Which people have Leo as a grandchild?
4. At step 1 after grabbing object 1, which facts were added, which were terminated, and which persisted?
5. After delivering object 1 to B at step 3, move the hand to C, grab object 2, move back to B, and release it. Check that object 1 remains at B throughout.
6. Suppose you manually retract `parent(ann, sam)` after running forward chaining. Will the already stored `grandparent(mary, sam)` disappear automatically? Why?
7. Does `backward_chain(..., ("at", "obj1", "B", 3))` automatically invent the three-action sequence? Explain the difference between the required Task-1 reasoner and the optional Task-2 planner.


## 18. Worked answers

**1.** With no previous bindings, the result is `{'?p': 'ann'}`. With `?p = mary`, it is `None` because the existing binding conflicts with `ann`.

**2.** `ask` does not consult rules. The initial memory contains only parent facts. Backward chaining uses the grandparent rule and establishes its two conditions.

**3.** Mary’s descendants under the ancestry rules are Ann, Sam, and Leo. Ann is Leo’s grandparent; Mary is Leo’s great-grandparent, for which we have no separate predicate. The following cell checks the two directions.


In [ ]:
extra_facts = FAMILY_FACTS + [("parent", "sam", "leo")]
fresh_forward = WorkingMemory(extra_facts)
fresh_backward = WorkingMemory(extra_facts)
forward_chain(fresh_forward, FAMILY_RULES)
q = ("ancestor", "mary", "?who")
fwd = {substitute(q, b) for b in fresh_forward.ask(q)}
bwd = {substitute(q, b) for b in backward_chain(fresh_backward, FAMILY_RULES, q)}
assert fwd == bwd
print("Mary's descendants:", sorted(fwd))
q = ("grandparent", "?who", "leo")
print("Leo's grandparents:", sorted({
    substitute(q, b) for b in backward_chain(fresh_backward, FAMILY_RULES, q)
}))


**4.** Added: `holding(obj1)` at step 1. Terminated for step 1: `empty(hand)` and `at(obj1, A)`. Persisted: the hand’s location, object 2’s location, and all object, location, and link facts. The step-0 facts stay in history.

**5.** The continuation below starts a fresh run, so it works even if you changed `world` during an exercise.


In [ ]:
exercise_world = WorkingMemory(WORLD_FACTS)
full_sequence = sequence + [
    (MOVE, ("move", "B", "C")),
    (GRAB, ("grab", "obj2", "C")),
    (MOVE, ("move", "C", "B")),
    (RELEASE, ("release", "obj2", "B")),
]
for t, (schema, action_name) in enumerate(full_sequence):
    selected = choose_action(exercise_world, schema, t, action_name)
    step(exercise_world, selected, t)
    if t + 1 >= 3:
        assert exercise_world.holds(("at", "obj1", "B", t + 1))

assert exercise_world.holds(("at", "obj1", "B", 7))
assert exercise_world.holds(("at", "obj2", "B", 7))
assert exercise_world.holds(("empty", "hand", 7))
print("At step 7 both objects are at B, and the hand is empty.")


**6.** No. This engine stores facts without dependency tracking. Removing a premise does not automatically retract its previously materialised consequences. Create fresh WM from the remaining base facts and derive again if you need conclusions based only on the remaining evidence.

**7.** No. Task-1 backward chaining establishes goals from existing facts and logical rules. It does not automatically implement action selection, persistence across hypothetical future states, or protection of earlier achieved goals. Those are additional planning requirements. After executing the sequence, a query about the stored step-3 fact can succeed by ordinary lookup.

## 19. Explain it in your own words

You should now be able to explain:

- How tuples represent facts, how variables appear in patterns, and how bindings connect rule conditions.
- Why forward chaining derives conclusions until a fixpoint, while backward chaining starts with a goal.
- How one recursive ancestry rule handles several generations.
- Why a logical conclusion and an action effect have different meanings.
- Why effects alone leave most of the next world unspecified, and how persistence fixes that in this model.
- Why unchanged facts must persist regardless of their predicate.
- How to keep domain knowledge in data and the reasoning engine generic.

**Source note:** based on the attached *Group Assignment – Knowledge Representation & Reasoning* and `mers_group_assignment_production_system_starter.py`. The family rules follow the assignment. Move/release schemas, the guarded grab schema, complete world data, explanations, and exercises are tutorial additions using the teacher’s domain.
